# Model Performance Comparison: Rule-Based vs CRF vs SciBERT vs GLiNER2

This notebook benchmarks four medical-dataset NER systems on two evaluation splits:

- **In-Distribution (ID)**: held-out test split from the same corpus used for training.
- **Out-of-Distribution (OOD)**: independently curated `151-eval.json` set (Elsevier ScienceDirect, 2025–2026) that no model has seen during training or hyperparameter tuning.

### Models compared

| Model | Type | Source notebook |
|---|---|---|
| **Rule-Based** | Pattern / gazetteer matcher | `Rule_based/medical_ner_rulebased.ipynb` |
| **CRF** | Linear-chain CRF (sklearn-crfsuite) | `CRF/crf_model.ipynb` |
| **SciBERT** | `allenai/scibert_scivocab_uncased` token classifier | `SciBERT/scibert_model.ipynb` |
| **GLiNER2** | `fastino/gliner2-large-v1` (full fine-tuned, 486M params) | `Gliner2/training_full.ipynb` |

### Evaluation methodology

All metrics use the same **chunk-level entity matching** protocol:

- **Exact match** — predicted span text matches a gold span exactly (case-insensitive, whitespace-normalized).
- **Partial match** — credited when prediction and gold span overlap (substring containment in either direction) but do not match exactly.

All figures are saved to `plots/` in PNG, PDF, and SVG formats at 300 DPI.

## Setup

Matplotlib configuration and colour palette. The style mirrors the original `performance-comparison.ipynb` (Times New Roman serif, 300 DPI, muted academic palette).

In [10]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Publication style (matches original performance-comparison settings)
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 11,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'figure.dpi': 300,
})

# Muted academic palette — one colour per model
C_RULE    = '#C98B7E'   # dusty rose  (rule-based)
C_CRF     = '#A8B4C4'   # steel gray  (CRF baseline)
C_SCIBERT = '#D6A75B'   # amber       (neural baseline)
C_GLINER  = '#2E6F8E'   # deep teal   (ours)
C_EDGE    = '#4A4A4A'

output_dir = Path('plots')
output_dir.mkdir(parents=True, exist_ok=True)

## Metrics

All numbers below are copied verbatim from each model's executed training/evaluation notebook. No aggregation, smoothing, or estimation is applied.

In [11]:
# Metrics extracted from the executed notebooks.
METRICS = {
    'RuleBased': {
        'ID':  {'exact_p': 0.3917, 'exact_r': 0.5661, 'exact_f1': 0.4630,
                'partial_p': 0.5079, 'partial_r': 0.7340, 'partial_f1': 0.6004},
        'OOD': {'exact_p': 0.5097, 'exact_r': 0.5685, 'exact_f1': 0.5375,
                'partial_p': 0.6520, 'partial_r': 0.7272, 'partial_f1': 0.6876},
    },
    'CRF': {
        'ID':  {'exact_p': 0.6851, 'exact_r': 0.6937, 'exact_f1': 0.6894,
                'partial_p': 0.7464, 'partial_r': 0.7558, 'partial_f1': 0.7511},
        'OOD': {'exact_p': 0.5601, 'exact_r': 0.3349, 'exact_f1': 0.4192,
                'partial_p': 0.6583, 'partial_r': 0.3936, 'partial_f1': 0.4927},
    },
    'SciBERT': {
        'ID':  {'exact_p': 0.8559, 'exact_r': 0.9424, 'exact_f1': 0.8971,
                'partial_p': 0.9046, 'partial_r': 0.9959, 'partial_f1': 0.9480},
        'OOD': {'exact_p': 0.7096, 'exact_r': 0.7111, 'exact_f1': 0.7104,
                'partial_p': 0.8468, 'partial_r': 0.8486, 'partial_f1': 0.8477},
    },
    'GLiNER2': {
        'ID':  {'exact_p': 0.7880, 'exact_r': 0.7975, 'exact_f1': 0.7927,
                'partial_p': 0.8554, 'partial_r': 0.8656, 'partial_f1': 0.8605},
        'OOD': {'exact_p': 0.7293, 'exact_r': 0.7509, 'exact_f1': 0.7400,
                'partial_p': 0.8162, 'partial_r': 0.8405, 'partial_f1': 0.8282},
    },
}

MODELS = ['RuleBased', 'CRF', 'SciBERT', 'GLiNER2']
MODEL_COLORS = {'RuleBased': C_RULE, 'CRF': C_CRF, 'SciBERT': C_SCIBERT, 'GLiNER2': C_GLINER}
MODEL_LABELS = {
    'RuleBased': 'Rule-Based',
    'CRF': 'CRF',
    'SciBERT': 'SciBERT',
    'GLiNER2': 'GLiNER2-Large',
}
MODEL_EDGE = {'RuleBased': C_EDGE, 'CRF': C_EDGE, 'SciBERT': C_EDGE, 'GLiNER2': '#1A3A4A'}
MODEL_ANNOT_COLOR = {'RuleBased': '#6A3A30', 'CRF': '#3A3A3A', 'SciBERT': '#6A4F1F', 'GLiNER2': '#1A3A4A'}

print('Metrics loaded for:', MODELS)

Metrics loaded for: ['RuleBased', 'CRF', 'SciBERT', 'GLiNER2']


### Plot helpers

Small utility functions reused by every figure — consistent axis styling, bar value labels, symmetric offsets for grouped bars, and a multi-format save routine.

In [12]:
def style_axes(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', alpha=0.3, linewidth=0.5, zorder=0)
    ax.xaxis.grid(False)
    ax.set_axisbelow(True)


def annotate_bars(ax, bars, color='#3A3A3A', offset=0.015, fontsize=8.5):
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + offset,
                f'{bar.get_height():.2f}', ha='center', va='bottom',
                fontsize=fontsize, fontweight='bold', color=color)


def save_all(fig, stem):
    for ext in ('png', 'pdf', 'svg'):
        fig.savefig(output_dir / f'{stem}.{ext}', dpi=300, bbox_inches='tight',
                    facecolor='white', edgecolor='none')
    print(f'Saved {stem}.(png|pdf|svg)')


def grouped_bar_offsets(n, width):
    """Return symmetric x-offsets centered on 0 for n bars of given width."""
    return np.array([(i - (n - 1) / 2) * width for i in range(n)])

## Figure 1 — In-Distribution Performance

Head-to-head **Exact** and **Partial** F1 on the held-out ID test split. This is the "home turf" comparison — every supervised model has seen data from the same distribution during training.

**What to look for:**

- Absolute ceiling each model reaches on familiar data.
- Spread between Exact and Partial F1 — a large gap indicates frequent boundary mistakes (the model finds the entity but clips it).
- Ordering of baselines relative to the neural models.

In [13]:
metrics_labels = ['Exact Match F1', 'Partial Match F1']
x = np.arange(len(metrics_labels))
width = 0.2
offsets = grouped_bar_offsets(len(MODELS), width)

fig, ax = plt.subplots(figsize=(8.2, 4.8))

for off, m in zip(offsets, MODELS):
    vals = [METRICS[m]['ID']['exact_f1'], METRICS[m]['ID']['partial_f1']]
    bars = ax.bar(x + off, vals, width, label=MODEL_LABELS[m],
                  color=MODEL_COLORS[m], edgecolor=MODEL_EDGE[m],
                  linewidth=0.6, zorder=3)
    annotate_bars(ax, bars, color=MODEL_ANNOT_COLOR[m])

ax.set_ylabel('F1 Score', fontsize=12, fontweight='bold', labelpad=8)
ax.set_xticks(x)
ax.set_xticklabels(metrics_labels, fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_yticks(np.arange(0, 1.1, 0.2))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.1f}'))
style_axes(ax)

ax.legend(loc='lower right', frameon=True, framealpha=0.95,
          edgecolor='#CCCCCC', fontsize=9, handlelength=1.5, ncol=2)
ax.set_title('Model Performance Comparison on In-Distribution Test Set',
             fontsize=12, fontweight='bold', pad=14, color='#1A1A1A')

plt.tight_layout()
save_all(fig, 'perfcomp_fig1_indistribution')

Saved perfcomp_fig1_indistribution.(png|pdf|svg)


## Figure 2 — Out-of-Distribution Performance

Same metrics (Exact + Partial F1), evaluated on the OOD set. None of the supervised models have seen these articles, author vocabularies, or venue styles during training — so this number reflects real-world deployment performance on fresh biomedical literature.

**What to look for:**

- How much each model's ranking *changes* relative to Figure 1.
- Which approach retains the most of its ID performance (generalization quality).
- Whether the rule-based baseline — having no training data to overfit to — is more stable on unfamiliar text.

In [15]:
fig, ax = plt.subplots(figsize=(8.2, 4.8))

for off, m in zip(offsets, MODELS):
    vals = [METRICS[m]['OOD']['exact_f1'], METRICS[m]['OOD']['partial_f1']]
    bars = ax.bar(x + off, vals, width, label=MODEL_LABELS[m],
                  color=MODEL_COLORS[m], edgecolor=MODEL_EDGE[m],
                  linewidth=0.6, zorder=3)
    annotate_bars(ax, bars, color=MODEL_ANNOT_COLOR[m])

ax.set_ylabel('F1 Score', fontsize=12, fontweight='bold', labelpad=8)
ax.set_xticks(x)
ax.set_xticklabels(metrics_labels, fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_yticks(np.arange(0, 1.1, 0.2))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.1f}'))
style_axes(ax)

ax.legend(loc='lower right', frameon=True, framealpha=0.95,
          edgecolor='#CCCCCC', fontsize=9, handlelength=1.5, ncol=2)
ax.set_title('Model Performance Comparison on Out-of-Distribution Test Set',
             fontsize=12, fontweight='bold', pad=14, color='#1A1A1A')

plt.tight_layout()
save_all(fig, 'perfcomp_fig2_outofdistribution')

Saved perfcomp_fig2_outofdistribution.(png|pdf|svg)


## Figure 3 — ID vs OOD Generalization Gap

Per-model ID and OOD bars side-by-side, for both Exact (left) and Partial (right) F1. A Δ% annotation above each pair quantifies the relative change on OOD: **green** when OOD actually *improves* over ID, **red** when it degrades.

This is the most direct view of **generalization quality** — a small or positive Δ means the model's inductive bias transfers; a large negative Δ means the model memorized training-set idiosyncrasies.

**What to look for:**

- Which model has the smallest drop? Smallest drop = best real-world deployment candidate.
- Does any model *improve* on OOD? That usually means the OOD set is easier or that the model was bottlenecked by training-set noise.
- Compare feature-engineered (CRF) vs pre-trained (SciBERT, GLiNER2) degradation patterns.

In [16]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8), sharey=True)

for ax, metric_key, title in zip(
    axes,
    ['exact_f1', 'partial_f1'],
    ['Exact Match F1', 'Partial Match F1'],
):
    xm = np.arange(len(MODELS))
    w = 0.36
    id_vals  = [METRICS[m]['ID'][metric_key]  for m in MODELS]
    ood_vals = [METRICS[m]['OOD'][metric_key] for m in MODELS]

    bars_id  = ax.bar(xm - w/2, id_vals,  w, label='In-Distribution',
                      color='#4A7FA8', edgecolor=C_EDGE, linewidth=0.6, zorder=3)
    bars_ood = ax.bar(xm + w/2, ood_vals, w, label='Out-of-Distribution',
                      color='#B5451B', edgecolor=C_EDGE, linewidth=0.6, zorder=3)

    annotate_bars(ax, bars_id,  color='#1A3A4A', fontsize=8.5)
    annotate_bars(ax, bars_ood, color='#6A2810', fontsize=8.5)

    for i, m in enumerate(MODELS):
        id_v, ood_v = id_vals[i], ood_vals[i]
        delta_pct = (ood_v - id_v) / id_v * 100
        sign = '+' if delta_pct >= 0 else '-'
        color = '#1A6A3A' if delta_pct >= 0 else '#6A2810'
        ymax = max(id_v, ood_v)
        ax.annotate(
            f'\u0394 {sign}{abs(delta_pct):.0f}%',
            xy=(xm[i], ymax + 0.08),
            ha='center', va='bottom',
            fontsize=8.5, fontstyle='italic', color=color, fontweight='bold',
        )

    ax.set_xticks(xm)
    ax.set_xticklabels([MODEL_LABELS[m] for m in MODELS], fontsize=9.5)
    ax.set_title(title, fontsize=11.5, fontweight='bold', pad=10)
    ax.set_ylim(0, 1.15)
    ax.set_yticks(np.arange(0, 1.1, 0.2))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.1f}'))
    style_axes(ax)
    ax.legend(loc='upper right', frameon=True, framealpha=0.95,
              edgecolor='#CCCCCC', fontsize=9, handlelength=1.5)

axes[0].set_ylabel('F1 Score', fontsize=12, fontweight='bold', labelpad=8)
fig.suptitle('In-Distribution vs Out-of-Distribution Generalization',
             fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
save_all(fig, 'perfcomp_fig3_id_vs_ood')

Saved perfcomp_fig3_id_vs_ood.(png|pdf|svg)


## Figure 4 — Precision / Recall / F1 Breakdown (Exact Match)

Disaggregates the single Exact-F1 number into its Precision and Recall components, with ID (left panel) and OOD (right panel) side-by-side. F1 alone can hide very different failure modes — e.g., high precision but low recall (conservative, misses entities) vs low precision but high recall (over-tags, produces many false positives).

**What to look for:**

- Does precision or recall collapse faster on OOD? That tells you whether novel text causes the model to hallucinate spans or to miss them.
- Rule-based / CRF precision vs SciBERT / GLiNER2 — pattern matching is usually the precision leader; neural models usually win on recall.
- Cases where one metric holds up and the other crashes (e.g., CRF's recall falls off a cliff on OOD while precision barely moves).

In [17]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.0), sharey=True)

pr_labels = ['Precision', 'Recall', 'F1']
pr_keys   = ['exact_p', 'exact_r', 'exact_f1']

for ax, split, split_title in zip(axes, ['ID', 'OOD'], ['In-Distribution', 'Out-of-Distribution']):
    xp = np.arange(len(pr_labels))
    wb = 0.2
    pr_offsets = grouped_bar_offsets(len(MODELS), wb)
    for off, m in zip(pr_offsets, MODELS):
        vals = [METRICS[m][split][k] for k in pr_keys]
        bars = ax.bar(xp + off, vals, wb, label=MODEL_LABELS[m],
                      color=MODEL_COLORS[m], edgecolor=MODEL_EDGE[m],
                      linewidth=0.6, zorder=3)
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.012,
                    f'{bar.get_height():.2f}', ha='center', va='bottom',
                    fontsize=7.5, fontweight='bold', color='#2A2A2A')

    ax.set_xticks(xp)
    ax.set_xticklabels(pr_labels, fontsize=10.5)
    ax.set_title(split_title, fontsize=11.5, fontweight='bold', pad=10)
    ax.set_ylim(0, 1.1)
    ax.set_yticks(np.arange(0, 1.1, 0.2))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.1f}'))
    style_axes(ax)

axes[0].set_ylabel('Score', fontsize=12, fontweight='bold', labelpad=8)
axes[1].legend(loc='lower right', frameon=True, framealpha=0.95,
               edgecolor='#CCCCCC', fontsize=8.5, handlelength=1.5, ncol=2)
fig.suptitle('Exact Match: Precision, Recall and F1 by Model and Split',
             fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
save_all(fig, 'perfcomp_fig4_prf_exact_breakdown')

Saved perfcomp_fig4_prf_exact_breakdown.(png|pdf|svg)


## Figure 5 — Consolidated Summary Table

Every metric for every (model, split) combination in one figure, rendered as a publication-ready table. Rows are grouped by model with soft background tints so adjacent ID/OOD rows for the same model read as a block.

Columns: Exact P / R / F1, Partial P / R / F1.

In [18]:
header = ['Model', 'Split', 'Exact P', 'Exact R', 'Exact F1',
          'Partial P', 'Partial R', 'Partial F1']
rows = []
for m in MODELS:
    for split in ['ID', 'OOD']:
        d = METRICS[m][split]
        rows.append([
            MODEL_LABELS[m], split,
            f"{d['exact_p']:.4f}",   f"{d['exact_r']:.4f}",   f"{d['exact_f1']:.4f}",
            f"{d['partial_p']:.4f}", f"{d['partial_r']:.4f}", f"{d['partial_f1']:.4f}",
        ])

fig, ax = plt.subplots(figsize=(11.0, 0.5 + 0.42 * (len(rows) + 1)), dpi=300)
ax.axis('off')

table = ax.table(cellText=rows, colLabels=header, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.0, 1.5)

for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor('black')
    cell.set_linewidth(0)
    if row == 0:
        cell.set_text_props(weight='bold')
        cell.visible_edges = 'BT'
        cell.set_linewidth(1.2)
    elif row == len(rows):
        cell.visible_edges = 'B'
        cell.set_linewidth(1.2)
    else:
        cell.visible_edges = ''

row_bg = {
    'Rule-Based': '#FAEFEC',
    'CRF': '#F5F7FA',
    'SciBERT': '#FBF4E5',
    'GLiNER2-Large': '#EAF1F6',
}
for r in range(1, len(rows) + 1):
    model_name = rows[r - 1][0]
    bg = row_bg.get(model_name, 'white')
    for c in range(len(header)):
        table[(r, c)].set_facecolor(bg)

plt.title('Consolidated Performance Metrics (ID + OOD)',
          fontsize=12, fontweight='bold', pad=10)
plt.tight_layout()
save_all(fig, 'perfcomp_fig5_summary_table')

Saved perfcomp_fig5_summary_table.(png|pdf|svg)


## Output summary

Lists every `perfcomp_*` file produced in `plots/` so you can sanity-check that all five figures were written in all three formats (PNG, PDF, SVG).

In [20]:
print('\nAll comparison figures saved to:', output_dir.resolve())
for p in sorted(output_dir.glob('perfcomp_*')):
    print(' -', p.name)


All comparison figures saved to: C:\Users\DELL\Desktop\NSU\Thesis\Gliner\MedNER\visualization\plots
 - perfcomp_fig1_indistribution.pdf
 - perfcomp_fig1_indistribution.png
 - perfcomp_fig1_indistribution.svg
 - perfcomp_fig2_outofdistribution.pdf
 - perfcomp_fig2_outofdistribution.png
 - perfcomp_fig2_outofdistribution.svg
 - perfcomp_fig3_id_vs_ood.pdf
 - perfcomp_fig3_id_vs_ood.png
 - perfcomp_fig3_id_vs_ood.svg
 - perfcomp_fig4_prf_exact_breakdown.pdf
 - perfcomp_fig4_prf_exact_breakdown.png
 - perfcomp_fig4_prf_exact_breakdown.svg
 - perfcomp_fig5_summary_table.pdf
 - perfcomp_fig5_summary_table.png
 - perfcomp_fig5_summary_table.svg
